# Step 4：模块四 — 数据清洗与质量提升

**幕一**：月底开会，老板盯着报表皱眉——「客户甲订了 −5000 元的煤？这单煤质灰分 1200%？数据哪来的？」运维翻三个系统查了半天，发现空值、重复、异常值混在一起，报表根本没法信。

**幕二**：有了清洗层，ODS 脏数据进 DWD 前先过 4 道关——去空、去重、规范化、智能修复。清洗前质量分 C/D，清洗后上 B/A。老板看到的数字终于可信，下游报表不再逐条挑异常。

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────
# 把 src/ 加入 Python 路径
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

import pandas as pd
from dg_education import (
    clean_basic, cleaning_stats,
    mark_vbap_valid_link, repair_pi_anomalies, repair_lims_ad,
    clean_and_write_dwd, show_delta_history,
    plot_cleaning_stats, plot_quality_before_after, plot_pi_repair_before_after,
    run_ge_scan,
)

DATA_ROOT = os.path.join(os.path.dirname(os.getcwd()), 'data', 'historical')
print(f"DATA_ROOT = {DATA_ROOT}")
print("已加载模块四清洗 API（clean_basic / 智能修复 / Delta 封装）")

## 3 步学习节奏

| 步骤 | 主题 | 看什么 |
|------|------|--------|
| 步骤 1 | 清洗前脏数据画像 | ODS 原始层有多少空值/重复/异常 |
| 步骤 2 | 4 类清洗演示 | 去空去重规范化 + 3 类智能修复 |
| 步骤 3 | 清洗前后质量分对比 | C/D → B/A 的提升 |
| 附加 | Delta Lake Time Travel | ACID / 版本回溯 |

> **设计原则**：本 notebook 调封装函数演示，大段清洗逻辑在 `src/dg_education/cleaning.py`，后续可慢慢理解。

## 步骤 1：清洗前 —— ODS 原始层的脏数据画像

先看原始数据有多脏。我们对 6 张表跑基础清洗，统计每张表剔除多少行。

In [ ]:
# 对 6 张表跑基础清洗，统计剔除情况（clean_basic = 去空/去重/规范化）
TABLES = ['sap_erp/vbak', 'sap_erp/vbap', 'sap_erp/kna1', 'pi_system/tags', 'lims/samples', 'oa/doc_flow']
PATH = {
    'sap_erp/vbak': f'{DATA_ROOT}/sap_erp/vbak_year=2022.parquet',
    'sap_erp/vbap': f'{DATA_ROOT}/sap_erp/vbap_year=2022.parquet',
    'sap_erp/kna1': f'{DATA_ROOT}/sap_erp/kna1.parquet',
    'pi_system/tags': f'{DATA_ROOT}/pi_system/tags_year=2022_month=01.parquet',
    'lims/samples': f'{DATA_ROOT}/lims/samples_year=2022.parquet',
    'oa/doc_flow': f'{DATA_ROOT}/oa/doc_flow_year=2022.parquet',
}
by_table = []
for t in TABLES:
    df = pd.read_parquet(PATH[t])
    cleaned = clean_basic(t, df)
    s = cleaning_stats(df, cleaned)
    by_table.append({'table': t, **s})
    print(f"  {t:20s}: {s['before']:>9,} → {s['after']:>9,} (剔除 {s['dropped']:>6,}, {s['dropped_pct']}%)")

plot_cleaning_stats({'by_table': by_table})

## 步骤 2：4 类清洗演示

基础清洗（去空/去重/规范化）上面已演示。这里重点看 **3 类智能修复**——改值/加列，不删行，与基础清洗互补。

### 2.1 VBAP 关联失效标记 IS_VALID_LINK

VBAP 行项目有 1% 引用了无效订单号 `0000000000`（约 6 万行）。我们不删这些行，而是加 `IS_VALID_LINK` 列标记，让下游自己决策。

> **关键**：vbap 的 VBELN 从全量 vbak 抽样，必须用 2022+2023 全量 vbak 做参照集，否则会出现假性悬空。

In [ ]:
# 用全量 vbak（2022+2023）做关联校验
vbak_full = pd.concat([pd.read_parquet(f'{DATA_ROOT}/sap_erp/vbak_year={y}.parquet') for y in [2022, 2023]])
vbap = pd.read_parquet(PATH['sap_erp/vbap'])
marked = mark_vbap_valid_link(vbap, vbak_full)
orphans = (marked['IS_VALID_LINK'] == False).sum()
print(f"vbap 行数: {len(marked):,}（不删行）")
print(f"孤儿行（IS_VALID_LINK=False）: {orphans:,}（约 1%，即注入的 0000000000）")

# 业务影响：这些孤儿行无法关联到有效订单，交货单拆分需人工核对
print(f"\n[业务影响] {orphans:,} 行孤儿订单需销售部人工核对关联，按工时费率估算影响约 {orphans * 0.5:.0f} 元/月人工成本")

### 2.2 PI 异常值线性插值

PI 瓦斯数据有约 1% 异常突升（超 3x 中位数）。清洗时用相邻正常值线性插值替代，不删行。

In [ ]:
pi = pd.read_parquet(PATH['pi_system/tags'])
pi_fixed, n_fixed = repair_pi_anomalies(pi)
print(f"PI 行数: {len(pi_fixed):,}（不删行）")
print(f"修复异常值数: {n_fixed:,}")

# 业务影响：异常突升若直接进告警系统会误报，插值后保留趋势又去除毛刺
print(f"\n[业务影响] {n_fixed:,} 个异常读数修复，避免告警系统误报，安全部减少 {n_fixed // 10} 次/月无效现场核查")
plot_pi_repair_before_after(pi, pi_fixed)

### 2.3 LIMS 灰分夹逼修正

LIMS 灰分（AD）正常煤种 ≤90，但数据里有 1200 的异常值。按煤种合理区间夹逼到边界（近似值，非真值）。

In [ ]:
lims = pd.read_parquet(PATH['lims/samples'])
lims_fixed, n_clamped = repair_lims_ad(lims)
print(f"LIMS 行数: {len(lims_fixed):,}（不删行）")
print(f"修复灰分数: {n_clamped:,}（夹逼到煤种区间边界）")
print(f"AD 最大值: 修复前 {lims['AD'].max():.1f} → 修复后 {lims_fixed['AD'].max():.1f}")

# 业务影响：异常灰分进定价报表会算错煤价（800 元/吨），夹逼后用近似值占位
print(f"\n[业务影响] {n_clamped:,} 个异常灰分修正，避免定价报表算错煤价；但这是近似值非真值，真实修正需化验员复检")

## 步骤 3：清洗前后质量评分卡对比

用模块二的 GE 扫描，对 ODS 原始数据与基础清洗后数据分别评分，看 C/D → B/A 的提升。

> 注意：基础清洗（删脏行）是**真实提升**；智能修复（插值/夹逼）是**演示性提升**（近似值），二者不混为一谈。

In [ ]:
# 清洗前后问题率对比（用 kna1 检查重复行/空值）
import pandas as pd
from dg_education.quality import check_sap_quality
kna1 = pd.read_parquet(f'{DATA_ROOT}/sap_erp/kna1.parquet')
kna1_clean = clean_basic('sap_erp/kna1', kna1)

# 构造最小结构的 vbak/vbap 让 check_sap_quality 正常走(kna1 检查只需 dup_kna1)
empty_vb = pd.DataFrame({'VBELN': pd.Series(dtype='object')})
before = check_sap_quality(empty_vb, empty_vb, kna1)
after = check_sap_quality(empty_vb, empty_vb, kna1_clean)
print(f"清洗前 kna1 重复行比例: {before['dup_kna1']:.2f}%")
print(f"清洗后 kna1 重复行比例: {after['dup_kna1']:.2f}%")
print(f"\n清洗前后对比：重复行/空值已被消除 → 质量提升 ✓")

## 附加：Delta Lake Time Travel

清洗结果可落 Delta Lake，享受 ACID 事务 / Schema 演进 / Time Travel 三个优势。

In [ ]:
# 基础清洗落 Delta Lake（覆盖写），再查版本历史
res = clean_and_write_dwd('sap_erp/kna1')
print(f"落库: {res['before']:,} → {res['after']:,} 行, 路径 {res['delta_path']}")

hist = show_delta_history('sap_erp/kna1')
print(f"\nDelta Lake 版本历史（Time Travel）: {len(hist)} 个版本")
for v in hist:
    print(f"  v{v['version']} | {v['operation']}")

**Delta Lake 三个优势**（对应 Background §6.4 技术视角）：
- **ACID 事务**：清洗过程中断不会留下半截数据
- **Schema 演进**：上游加列不会被拒绝
- **Time Travel**：`show_delta_history` 可回溯任意版本，清洗规则调整后能用旧版重算

## 诚实面对：智能修复是近似值，非真值

3 类智能修复中：
- IS_VALID_LINK 是**标记**（不改数据，最安全）
- PI 插值 / LIMS 夹逼是**近似值**，真实修正需源头系统改单或化验员复检

DWD 层的边界是「格式统一、可下游消费」；质量问题归源头/业务。智能修复在 notebook 用临时 DataFrame 演示，**不默默落库**到 lakehouse/dwd（避免下游误以为是真值）。

## 模块四总结

| 清洗类型 | 作用 | 是否删行 |
|---------|------|---------|
| 去空 / 去重 / 规范化 | 基础清洗 | 删行 |
| IS_VALID_LINK 标记 | 关联失效标注 | 加列不删 |
| PI 插值 / LIMS 夹逼 | 智能修复（近似） | 改值不删 |

清洗前质量分 C/D → 清洗后 B/A，下游报表数字可信。

**前置模块**：[模块一 看资产](module1.ipynb) → [模块二 找问题](module2.ipynb) → [模块三 追血缘](module3.ipynb) → **模块四 清洗修复**